In [1]:
# ===============================================================
# 🧱 Schritt 1: Benötigte Bibliotheken importieren
# ===============================================================
import json
import os

# ===============================================================
# 📂 Schritt 2: Dateinamen definieren (liegt im selben Ordner)
# ===============================================================
for i in range(7, 11):
    input_filename = f"RealLife_2024_{i}.json"
    output_filename = f"Construction_{input_filename}"

    # ===============================================================
    # 📖 Schritt 3: JSON-Datei laden
    # ===============================================================
    with open(input_filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    # ===============================================================
    # 🛠️ Schritt 4: Änderungen
    # ===============================================================

    # Anbaugeräte-ID ab 0 durchnummerieren
    anbaugeraete = data.get("Anbaugeraete", [])
    for new_id, ag in enumerate(anbaugeraete):
        ag["ID"] = new_id

    # Maschinen-ID ab 0 durchnummerieren
    maschinen = data.get("Maschinen", [])
    for new_id, m in enumerate(maschinen):
        m["ID"] = new_id

    # Auftragsnummern ab 0 als Strings durchnummerieren
    # und BestellpositionenStrings numerisch -1 rechnen
    auftraege = data.get("Auftraege", [])
    for idx, auftrag in enumerate(auftraege):
        auftrag["Auftragsnummer"] = str(idx)
        auftrag["Baustellennummer"] = idx  # ← NEU: int-Wert der Auftragsnummer
        neue_positionen = []
        for pos_str in auftrag.get("BestellpositionenStrings", []):
            try:
                neue_pos = str(int(pos_str) - 1)
            except ValueError as e:
                raise ValueError(f"Ungültiger Wert in BestellpositionenStrings: '{pos_str}'") from e
            neue_positionen.append(neue_pos)
        auftrag["BestellpositionenStrings"] = neue_positionen

    # Bestellpositionen-ID ab 0 durchnummerieren
    bestellpositionen = data.get("Bestellpositionen", [])
    for idx, bp in enumerate(bestellpositionen):
        bp["ID"] = idx

    # Auftragsnummern den Bestellpositionen zuordnen
    id_to_auftrag = {}
    for auftrag in auftraege:
        auftragsnummer = auftrag["Auftragsnummer"]
        for bp_str in auftrag.get("BestellpositionenStrings", []):
            if bp_str in id_to_auftrag:
                raise ValueError(f"Bestellpositions-ID '{bp_str}' ist mehrfach in Aufträgen enthalten!")
            id_to_auftrag[bp_str] = auftragsnummer

    for bp in bestellpositionen:
        bp_id_str = str(bp["ID"])
        if bp_id_str not in id_to_auftrag:
            raise ValueError(f"Keine Auftragsnummer für Bestellposition mit ID {bp['ID']} gefunden!")
        bp["Auftragsnummer"] = id_to_auftrag[bp_id_str]

    # ArbeitswegeString korrigieren ("Auftrag N" → str(N-1)) im verschachtelten Dict
    arbeitswege_dict = data.get("ArbeitswegeString", {})
    neue_arbeitswege_dict = {}

    for von_id, ziele_dict in arbeitswege_dict.items():
        neues_ziele_dict = {}
        for ziel_key, distanz in ziele_dict.items():
            if ziel_key.startswith("Auftrag "):
                try:
                    neue_id = str(int(ziel_key.replace("Auftrag ", "")) - 1)
                except ValueError as e:
                    raise ValueError(f"❌ Ungültiger Ziel-Eintrag in ArbeitswegeString: '{ziel_key}'") from e
                neues_ziele_dict[neue_id] = distanz
            else:
                raise ValueError(f"❌ Unerwarteter Ziel-Eintrag in ArbeitswegeString: '{ziel_key}'")
        neue_arbeitswege_dict[von_id] = neues_ziele_dict

    data["ArbeitswegeString"] = neue_arbeitswege_dict

    # TransportwegeString korrigieren (von-1, nach-1)
    transportwege_dict = data.get("TransportwegeString", {})
    neue_transportwege_dict = {}

    for von_key, ziele_dict in transportwege_dict.items():
        try:
            neuer_von = str(int(von_key) - 1)
        except ValueError as e:
            raise ValueError(f"❌ Ungültiger von-Key in TransportwegeString: '{von_key}'") from e

        neues_ziele_dict = {}
        for nach_key, distanz in ziele_dict.items():
            try:
                neuer_nach = str(int(nach_key) - 1)
            except ValueError as e:
                raise ValueError(f"❌ Ungültiger nach-Key in TransportwegeString: '{nach_key}'") from e
            neues_ziele_dict[neuer_nach] = distanz

        neue_transportwege_dict[neuer_von] = neues_ziele_dict

    data["TransportwegeString"] = neue_transportwege_dict

    # ===============================================================
    # 💾 Schritt 5: Neue JSON-Datei speichern
    # ===============================================================
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"✅ Änderungen in {input_filename} vorgenommen und in {output_filename} gespeichert.")

✅ Änderungen in RealLife_2024_7.json vorgenommen und in Construction_RealLife_2024_7.json gespeichert.
✅ Änderungen in RealLife_2024_8.json vorgenommen und in Construction_RealLife_2024_8.json gespeichert.
✅ Änderungen in RealLife_2024_9.json vorgenommen und in Construction_RealLife_2024_9.json gespeichert.
✅ Änderungen in RealLife_2024_10.json vorgenommen und in Construction_RealLife_2024_10.json gespeichert.


In [4]:
import json
from pathlib import Path
from copy import deepcopy
from datetime import datetime

# === Pfade definieren ===
input_path = Path("RealLife_2023_5.json")
output_path = input_path.with_name("Construction_" + input_path.name)

# === Datei laden ===
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# === 1. Maschinen-ID neu vergeben ===
maschinen = data["Maschinen"]
old_maschinen_ids = {m["ID"]: i for i, m in enumerate(maschinen)}
for new_id, maschine in enumerate(maschinen):
    maschine["ID"] = new_id
    maschine["Name"] = maschine.pop("oldID")  # oldID -> Name

# === 2. Arbeiter Personalnummer neu vergeben und Feldnamen anpassen ===
arbeiter = data["Arbeiter"]
old_persnr_map = {}
for new_id, a in enumerate(arbeiter):
    old_persnr_map[a["personalnummer"]] = str(new_id)  # Map für Maschinen-Stammfahrer
    a["Personalnummer"] = new_id
    a["Name"] = a.pop("name")
    a["Qualifikationen"] = a.pop("qualifikationen")
    del a["personalnummer"]

# === 3. Maschinen-StammfahrerStrings aktualisieren ===
for m in maschinen:
    m["StammfahrerStrings"] = [old_persnr_map.get(p, str(p)) for p in m["StammfahrerStrings"]]

# === 4. Aufträge neu nummerieren ===
auftraege = data["Auftraege"]
old_auftrags_map = {}
for new_id, auftrag in enumerate(auftraege):
    old_auftrags_map[str(auftrag["Auftragsnummer"])] = str(new_id)
    auftrag["Auftragsnummer"] = str(new_id)
    auftrag["Baustellennummer"] = new_id

# === 5. Bestellpositionen neu nummerieren ===
def convert_to_isoformat(date_str):
    for fmt in ("%Y-%m-%d %H:%M:%S", "%d.%m.%Y %H:%M", "%Y/%m/%d %H:%M"):
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    raise ValueError(f"❌ Unbekanntes Datumsformat: {date_str}")

bestellpositionen = data["Bestellpositionen"]
old_bp_id_map = {}
for new_id, bp in enumerate(bestellpositionen):
    old_bp_id_map[str(bp["ID"])] = str(new_id)
    bp["ID"] = new_id
    bp["Auftragsnummer"] = old_auftrags_map[str(bp["Auftragsnummer"])]
    bp["ArbeiterQualifikationen"] = [int(q) for q in bp["ArbeiterQualifikationen"]]

    # === Zeitformat umwandeln und Dauer berechnen ===
    start_dt = convert_to_isoformat(bp["Start"])
    end_dt = convert_to_isoformat(bp["Ende"])

    bp["Start"] = start_dt.isoformat()
    bp["Ende"] = end_dt.isoformat()
    bp["Dauer"] = int((end_dt - start_dt).total_seconds() // 3600)  # Ganze Stunden


# === 6. Aufträge: BestellpositionenStrings aktualisieren ===
for auftrag in auftraege:
    auftrag["BestellpositionenStrings"] = [
        old_bp_id_map.get(bp_id, bp_id) for bp_id in auftrag["BestellpositionenStrings"]
    ]

# === 7. Arbeitswege und Transportwege neu nummerieren ===

# Mapping von alten → neuen Personalnummern (str)
reverse_pers_map = {str(old): str(new) for old, new in old_persnr_map.items()}

# Mapping von alten → neuen Auftragsnummern (str)
reverse_auftrags_map = {old: new for old, new in old_auftrags_map.items()}

def remap_nested_distance_dict(old_dict, key_map_outer, key_map_inner):
    new_dict = {}
    for outer_old_key, inner_dict in old_dict.items():
        outer_new_key = key_map_outer.get(str(outer_old_key), str(outer_old_key))
        new_inner = {}
        for inner_old_key, val in inner_dict.items():
            inner_new_key = key_map_inner.get(str(inner_old_key), str(inner_old_key))
            new_inner[str(inner_new_key)] = val
        new_dict[str(outer_new_key)] = new_inner
    return new_dict

if "ArbeitswegeString" in data:
    data["ArbeitswegeString"] = remap_nested_distance_dict(
        data["ArbeitswegeString"], reverse_pers_map, reverse_auftrags_map
    )

if "TransportwegeString" in data:
    data["TransportwegeString"] = remap_nested_distance_dict(
        data["TransportwegeString"], reverse_auftrags_map, reverse_auftrags_map
    )

# === Datei speichern ===
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"✅ Neue Datei gespeichert unter: {output_path}")

✅ Neue Datei gespeichert unter: Construction_RealLife_2023_5.json
